# 실습 4회: Qwen3-0.6B 파인튜닝 — 재무 Q&A 모델 학습

서울대학교 핀테크 실습 과정 | 12기

---

## 학습 목표

1. LLM 토크나이저(Tokenizer)의 동작 원리와 Qwen3의 특수 토큰 체계를 이해한다.
2. 3회차에서 추출한 삼성전자 감사보고서 데이터를 SFT 학습용 Q&A 데이터셋으로 변환한다.
3. QLoRA (4-bit 양자화 + LoRA)를 적용하여 Qwen3-0.6B를 재무 도메인에 파인튜닝한다.
4. 파인튜닝 전후 모델의 답변을 비교하고 Thinking / Non-thinking 모드를 실습한다.

---

## 사용 모델 및 데이터

| 항목 | 내용 |
|:---|:---|
| 베이스 모델 | `Qwen/Qwen3-0.6B` (0.6B 파라미터, 28층, context 32k) |
| 학습 방식 | SFT (Supervised Fine-Tuning) + QLoRA |
| 학습 데이터 | 삼성전자 2024년 별도 재무제표 Q&A (3회차 추출 결과 활용) |
| 실행 환경 | Google Colab T4 GPU (무료) |

---

## 전체 파이프라인

```
[3회차 출력물]
  재무제표 CSV (BS / IS / OCI / CSE / CF)
  sections.json (섹션 텍스트)
        |
[Step 3] 토크나이저 이해
  BPE / 특수 토큰 / Chat Template / Thinking 모드
        |
[Step 4] Q&A 데이터셋 자동 생성
  수치 조회 / 비율 계산 / 증감 분석 / 텍스트 Q&A
        |
[Step 5] Qwen3-0.6B QLoRA 파인튜닝
  4-bit 양자화 + LoRA 어댑터 학습 (SFTTrainer)
        |
[Step 6] 추론 및 모드 비교
  파인튜닝 전후 / Thinking vs Non-thinking
```


---

## Step 1. 환경 감지 및 패키지 설치

이 노트북은 **세 가지 실행 환경**을 자동 감지하고 설정을 분기한다.

| 환경 | 특징 | 양자화 | 학습 속도 |
|:---|:---|:---:|:---:|
| **Google Colab (T4 GPU)** | NVIDIA CUDA, 무료 T4 15 GB | 4-bit QLoRA 가능 | 빠름 |
| **로컬 — CUDA GPU** | NVIDIA GPU 장착 PC | 4-bit QLoRA 가능 | 빠름 |
| **로컬 — Mac (Apple Silicon)** | MPS 백엔드, bfloat16 | QLoRA 불가 → float16 | 보통 |

> `bitsandbytes` 라이브러리는 CUDA 전용이다. Mac / CPU 환경에서는 4-bit 양자화를 건너뛰고
> `torch_dtype=torch.float16` 으로 전체 가중치를 로드한다.

### 필요 패키지

| 패키지 | 용도 | CUDA | Mac/CPU |
|:---|:---|:---:|:---:|
| `transformers` (>=4.51) | 모델·토크나이저 로드 | O | O |
| `trl` (>=0.15) | SFTTrainer | O | O |
| `peft` (>=0.14) | LoRA 어댑터 | O | O |
| `bitsandbytes` (>=0.43) | 4-bit 양자화 | O | 선택 |
| `datasets` | HuggingFace Dataset | O | O |
| `accelerate` | GPU/MPS 설정 | O | O |


In [1]:
import sys, subprocess, platform

# ── 환경 사전 감지 ──────────────────────────────────────────
try:
    import google.colab; _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

_IS_MAC = (platform.system() == 'Darwin')

# ── 패키지 설치 ─────────────────────────────────────────────
# Colab 또는 패키지가 없는 로컬 환경에서 실행
# 로컬에 이미 설치되어 있다면 이 셀은 건너뛰어도 된다

BASE_PKGS = [
    'transformers>=4.51.0',
    'trl>=0.15.0',
    'peft>=0.14.0',
    'datasets',
    'accelerate',
]

if _IN_COLAB:
    # Colab: 항상 설치 (환경이 초기화되므로)
    pkgs = BASE_PKGS + ['bitsandbytes>=0.43.0']
    print(f'[Colab] 패키지 설치 중: {pkgs}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
elif _IS_MAC:
    # Mac: bitsandbytes 는 CUDA 전용이므로 제외, 나머지만 설치
    print('[Mac] bitsandbytes 제외 후 설치 (MPS 환경에서는 4-bit 양자화 미지원)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + BASE_PKGS, check=True)
    # Mac 에서 bitsandbytes 를 억지로 설치해도 CPU 전용 fallback 만 동작함
else:
    # 로컬 CUDA: 필요 시 설치
    pkgs = BASE_PKGS + ['bitsandbytes>=0.43.0']
    print(f'[로컬 CUDA] 패키지 확인/설치 중...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)

print('패키지 설치 완료')


[Mac] bitsandbytes 제외 후 설치 (MPS 환경에서는 4-bit 양자화 미지원)
패키지 설치 완료


In [2]:
import torch, platform

# ── 실행 환경 및 디바이스 감지 ──────────────────────────────
try:
    import google.colab; IN_COLAB = True
except ImportError:
    IN_COLAB = False

IS_MAC = (platform.system() == 'Darwin')

if torch.cuda.is_available():
    DEVICE      = 'cuda'
    USE_4BIT    = True                     # QLoRA 가능
    TORCH_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    BF16_OK     = torch.cuda.is_bf16_supported()
    OPTIM       = 'paged_adamw_8bit'       # bitsandbytes 8-bit 옵티마이저
elif IS_MAC and hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE      = 'mps'
    USE_4BIT    = False                    # bitsandbytes 미지원
    TORCH_DTYPE = torch.float16            # MPS 에서 bfloat16 부분 지원
    COMPUTE_DTYPE = torch.float16
    BF16_OK     = False
    OPTIM       = 'adamw_torch'
else:
    DEVICE      = 'cpu'
    USE_4BIT    = False
    TORCH_DTYPE = torch.float32
    COMPUTE_DTYPE = torch.float32
    BF16_OK     = False
    OPTIM       = 'adamw_torch'

ENV_CONFIG = {
    'in_colab':      IN_COLAB,
    'is_mac':        IS_MAC,
    'device':        DEVICE,
    'use_4bit':      USE_4BIT,
    'torch_dtype':   TORCH_DTYPE,
    'compute_dtype': COMPUTE_DTYPE,
    'bf16_ok':       BF16_OK,
    'optim':         OPTIM,
}

# ── 출력 ────────────────────────────────────────────────────
env_label = 'Google Colab' if IN_COLAB else ('Mac (Apple Silicon)' if IS_MAC else '로컬 CUDA')
print(f'실행 환경   : {env_label}')
print(f'디바이스    : {DEVICE.upper()}')
print(f'torch dtype : {TORCH_DTYPE}')
print(f'4-bit QLoRA : {"사용" if USE_4BIT else "미사용 (float16/float32 로드)"}')
print(f'bf16 지원   : {BF16_OK}')
print(f'옵티마이저  : {OPTIM}')
print()

if DEVICE == 'cuda':
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU  : {gpu}')
    print(f'VRAM : {vram:.1f} GB')
elif DEVICE == 'mps':
    mem = torch.mps.recommended_max_memory() / 1e9 if hasattr(torch.mps, 'recommended_max_memory') else '?'
    print(f'MPS (Apple Silicon) | 통합 메모리: {mem:.0f} GB' if isinstance(mem, float) else 'MPS (Apple Silicon)')
    print('주의: 4-bit 양자화 미사용. 학습 속도가 CUDA 대비 느릴 수 있습니다.')
else:
    print('CPU 모드. 소규모 테스트 전용 — 학습 시간이 매우 길어집니다.')


실행 환경   : Mac (Apple Silicon)
디바이스    : MPS
torch dtype : torch.float16
4-bit QLoRA : 미사용 (float16/float32 로드)
bf16 지원   : False
옵티마이저  : adamw_torch

MPS (Apple Silicon) | 통합 메모리: 206 GB
주의: 4-bit 양자화 미사용. 학습 속도가 CUDA 대비 느릴 수 있습니다.


---

## Step 2. 파일 준비 및 경로 설정

3회차에서 생성한 출력 파일들을 환경에 맞는 경로로 준비한다.

| 환경 | 파일 위치 방법 | 기본 경로 |
|:---|:---|:---|
| **Colab** | 직접 업로드 또는 Google Drive 마운트 | `/content/` |
| **로컬 CUDA** | 3회차 실습과 동일 디렉터리 | `../output/processed/tables/2024/` |
| **Mac** | 3회차 실습과 동일 디렉터리 | `../output/processed/tables/2024/` |

**Colab 업로드 방법:**

방법 A — 직접 업로드
```python
from google.colab import files
uploaded = files.upload()  # CSV / JSON 파일 선택
```

방법 B — Google Drive 마운트 (권장, 파일이 많을 때)
```python
from google.colab import drive
drive.mount('/content/drive')
# 이후 TABLES_DIR 을 Drive 경로로 변경:
# TABLES_DIR = Path('/content/drive/MyDrive/12기핀테크실습/output/...')
```

### 필요 파일

| 파일 | 설명 |
|:---|:---|
| `2024_BS_재무상태표.csv` | 재무상태표 |
| `2024_IS_손익계산서.csv` | 손익계산서 |
| `2024_CF_현금흐름표.csv` | 현금흐름표 |
| `2024_CSE_자본변동표.csv` | 자본변동표 |
| `sections.json` | 섹션 텍스트 (감사 의견 등) |


In [3]:
import os, json
from pathlib import Path

# ENV_CONFIG 는 앞 셀에서 이미 설정됨
# ─────────────────────────────────────
# 경로 설정 (환경에 따라 자동 분기)
# ─────────────────────────────────────
if ENV_CONFIG['in_colab']:
    # Colab: /content 에 파일을 업로드하거나 Drive 를 마운트 후 경로 변경
    TABLES_DIR    = Path('/content')
    SECTIONS_PATH = Path('/content/sections.json')
    OUTPUT_DIR    = Path('/content/finetune')
    print('[Colab] 파일을 /content 에 업로드하거나 Drive 마운트 후 경로를 수정하세요.')

elif ENV_CONFIG['device'] == 'mps':  # Mac
    TABLES_DIR    = Path('../output/processed/tables/2024')
    SECTIONS_PATH = Path('../output/processed/jsons/2024/sections.json')
    OUTPUT_DIR    = Path('../output/finetune')
    print('[Mac] 로컬 출력 디렉터리를 사용합니다.')
    print('      학습은 MPS 로 실행됩니다 (float16, 양자화 없음).')

else:  # 로컬 CUDA
    TABLES_DIR    = Path('../output/processed/tables/2024')
    SECTIONS_PATH = Path('../output/processed/jsons/2024/sections.json')
    OUTPUT_DIR    = Path('../output/finetune')
    print(f'[로컬 CUDA] 로컬 출력 디렉터리를 사용합니다.')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'  TABLES_DIR    : {TABLES_DIR.resolve()}')
print(f'  SECTIONS_PATH : {SECTIONS_PATH.resolve()}')
print(f'  OUTPUT_DIR    : {OUTPUT_DIR.resolve()}')
print()

# ── 파일 존재 확인 ──────────────────────────────────────────
required = [
    TABLES_DIR / '2024_BS_재무상태표.csv',
    TABLES_DIR / '2024_IS_손익계산서.csv',
    TABLES_DIR / '2024_CF_현금흐름표.csv',
    TABLES_DIR / '2024_CSE_자본변동표.csv',
    SECTIONS_PATH,
]
all_ok = True
for p in required:
    ok = p.exists()
    if not ok: all_ok = False
    print(f'  [{"OK" if ok else "없음"}] {p.name}')
if not all_ok:
    print('\n일부 파일 없음')
    if ENV_CONFIG['in_colab']:
        print('  -> Colab: 위 셀을 참고하여 파일을 업로드하세요.')
    else:
        print('  -> 로컬: 3회차 노트북을 먼저 실행하여 CSV/JSON 을 생성하세요.')
else:
    print('모든 파일 확인 완료')


[Mac] 로컬 출력 디렉터리를 사용합니다.
      학습은 MPS 로 실행됩니다 (float16, 양자화 없음).
  TABLES_DIR    : /Users/dongjunjang/Desktop/12기핀테크실습/output/processed/tables/2024
  SECTIONS_PATH : /Users/dongjunjang/Desktop/12기핀테크실습/output/processed/jsons/2024/sections.json
  OUTPUT_DIR    : /Users/dongjunjang/Desktop/12기핀테크실습/output/finetune

  [OK] 2024_BS_재무상태표.csv
  [OK] 2024_IS_손익계산서.csv
  [OK] 2024_CF_현금흐름표.csv
  [OK] 2024_CSE_자본변동표.csv
  [OK] sections.json
모든 파일 확인 완료


---

## Step 3. 토크나이저(Tokenizer) 이해

### 3.1 토크나이저란

LLM은 텍스트를 직접 처리하지 않고 **정수 ID 시퀀스**로 변환한 뒤 처리한다.
이 변환을 담당하는 모듈이 **토크나이저(Tokenizer)**이다.

```
'삼성전자의 매출액은?'  -->  [토크나이저]  -->  [34567, 12890, 45231, 88901]
                                              토큰 ID 시퀀스
```

---

### 3.2 BPE (Byte Pair Encoding)

Qwen3는 **BPE** 방식으로 어휘를 구성한다.
학습 코퍼스에서 자주 함께 등장하는 문자 쌍을 반복 병합하여 어휘를 만든다.

```
초기  : '삼' '성' '전' '자'    (글자 단위)
병합1 : '삼성' '전' '자'
병합2 : '삼성' '전자'
결과  : '삼성전자'              (하나의 토큰)
```

- Qwen3 어휘 크기: **151,936개** 토큰
- 영어는 단어/서브워드 단위, 한국어는 음절 또는 형태소 단위로 분리

---

### 3.3 특수 토큰 (Special Tokens)

LLM은 구조적 의미를 가진 특수 토큰을 사용한다.

| 토큰 | ID | 의미 |
|:---|:---:|:---|
| `<\|endoftext\|>` | 151643 | 텍스트 종료 (EOS) |
| `<\|im_start\|>` | 151644 | 대화 턴 시작 |
| `<\|im_end\|>` | 151645 | 대화 턴 종료 |
| `<think>` | 151667 | Thinking 구간 시작 |
| `</think>` | 151668 | Thinking 구간 종료 |

---

### 3.4 Chat Template

Qwen3의 Chat Template은 role별 메시지를 다음 구조로 변환한다.

```
<|im_start|>system
당신은 재무 전문가입니다.<|im_end|>
<|im_start|>user
매출액은 얼마인가요?<|im_end|>
<|im_start|>assistant
<think>
...(사고 과정)...
</think>
답변 내용<|im_end|>
```

---

### 3.5 Thinking 모드 vs Non-thinking 모드

Qwen3의 핵심 기능: **하나의 모델에서 두 가지 추론 모드**를 지원한다.

| 모드 | `enable_thinking` | 특징 | 권장 상황 |
|:---|:---:|:---|:---|
| **Thinking** | `True` (기본) | `<think>` 사고 후 답변 | 수학·코딩·복잡 추론 |
| **Non-thinking** | `False` | 즉시 답변 | 단순 Q&A, 빠른 응답 |

**권장 샘플링 파라미터:**
- Thinking: `Temperature=0.6, TopP=0.95, TopK=20`
- Non-thinking: `Temperature=0.7, TopP=0.8, TopK=20`

> Greedy decoding (do_sample=False 또는 temperature=0)은 사용하지 말 것.
> Qwen3 공식 문서에서 반복 출력 문제가 발생할 수 있다고 명시하고 있다.


In [4]:
# [실습 3-1] Qwen3 토크나이저 로드 및 기본 정보
from transformers import AutoTokenizer

MODEL_NAME = 'Qwen/Qwen3-0.6B'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

print('[토크나이저 기본 정보]')
print(f'  어휘 크기 (vocab_size): {tokenizer.vocab_size:,}')
print(f'  최대 길이:              {tokenizer.model_max_length:,} 토큰')
print(f'  EOS 토큰:               {tokenizer.eos_token!r}')
print(f'  PAD 토큰:               {tokenizer.pad_token!r}')


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

[토크나이저 기본 정보]
  어휘 크기 (vocab_size): 151,643
  최대 길이:              131,072 토큰
  EOS 토큰:               '<|im_end|>'
  PAD 토큰:               '<|endoftext|>'


In [5]:
# [실습 3-2] 한국어 / 영어 토큰화 비교
# 같은 의미의 문장이 언어에 따라 토큰 수가 다름을 확인한다

test_cases = [
    ('KO', '삼성전자의 2024년 매출액은 209조원입니다.'),
    ('EN', 'Samsung Electronics revenue in 2024 was 209 trillion KRW.'),
    ('KO', '영업이익률'),
    ('KO', 'Ⅰ. 유 동 자 산'),
    ('KO', '자산총계: 324,966,127 백만원'),
]

print(f'{"언어":<4} {"토큰수":>6}  처음 6개 토큰')
print('-' * 65)
for lang, text in test_cases:
    ids    = tokenizer.encode(text, add_special_tokens=False)
    tokens = tokenizer.convert_ids_to_tokens(ids[:6])
    print(f'{lang:<4} {len(ids):>6}  {tokens}')
    print(f'       원문: {text[:55]}')
    print()


언어      토큰수  처음 6개 토큰
-----------------------------------------------------------------
KO       22  ['ìĤ¼', 'ìĦ±', 'ìłĦ', 'ìŀĲìĿĺ', 'Ġ', '2']
       원문: 삼성전자의 2024년 매출액은 209조원입니다.

EN       18  ['Samsung', 'ĠElectronics', 'Ġrevenue', 'Ġin', 'Ġ', '2']
       원문: Samsung Electronics revenue in 2024 was 209 trillion KR

KO        5  ['ìĺģ', 'ìĹħ', 'ìĿ´', 'ìĿµ', 'ë¥ł']
       원문: 영업이익률

KO        8  ['âħ', 'ł', '.', 'Ġìľł', 'ĠëıĻ', 'ĠìŀĲ']
       원문: Ⅰ. 유 동 자 산

KO       21  ['ìŀĲ', 'ìĤ°', 'ì´Ŀ', 'ê³Ħ', ':', 'Ġ']
       원문: 자산총계: 324,966,127 백만원



In [6]:
# [실습 3-3] 특수 토큰 확인

special_tokens = [
    ('<|endoftext|>', '텍스트 종료 (EOS)'),
    ('<|im_start|>',  '대화 턴 시작'),
    ('<|im_end|>',    '대화 턴 종료'),
    ('<think>',       'Thinking 구간 시작'),
    ('</think>',      'Thinking 구간 종료'),
]

print(f'{"토큰":<20} {"ID":>8}  설명')
print('-' * 55)
for tok, desc in special_tokens:
    tid = tokenizer.convert_tokens_to_ids(tok)
    print(f'{tok:<20} {tid:>8}  {desc}')

print()
print('[ID -> 토큰 역방향 확인]')
for tid in [151643, 151644, 151645, 151667, 151668]:
    print(f'  {tid} -> {tokenizer.decode([tid])!r}')


토큰                         ID  설명
-------------------------------------------------------
<|endoftext|>          151643  텍스트 종료 (EOS)
<|im_start|>           151644  대화 턴 시작
<|im_end|>             151645  대화 턴 종료
<think>                151667  Thinking 구간 시작
</think>               151668  Thinking 구간 종료

[ID -> 토큰 역방향 확인]
  151643 -> '<|endoftext|>'
  151644 -> '<|im_start|>'
  151645 -> '<|im_end|>'
  151667 -> '<think>'
  151668 -> '</think>'


In [7]:
# [실습 3-4] Chat Template 동작 확인
# messages 딕셔너리가 실제 모델 입력 텍스트로 어떻게 변환되는지 확인한다

sample_messages = [
    {'role': 'system',    'content': '당신은 삼성전자 재무제표를 분석하는 전문가입니다.'},
    {'role': 'user',      'content': '2024년 매출액은 얼마인가요?'},
    {'role': 'assistant', 'content': '209조 524억원(209,052,241 백만원)입니다.'},
]

text = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=False,
    enable_thinking=False,
)

print('[Non-thinking 모드 Chat Template 결과]')
print(text)
print(f'총 토큰 수: {len(tokenizer.encode(text))}')


[Non-thinking 모드 Chat Template 결과]
<|im_start|>system
당신은 삼성전자 재무제표를 분석하는 전문가입니다.<|im_end|>
<|im_start|>user
2024년 매출액은 얼마인가요?<|im_end|>
<|im_start|>assistant
<think>

</think>

209조 524억원(209,052,241 백만원)입니다.<|im_end|>

총 토큰 수: 82


In [8]:
# [실습 3-5] Thinking / Non-thinking 모드 Chat Template 비교

gen_messages = [
    {'role': 'system', 'content': '당신은 재무 전문가입니다.'},
    {'role': 'user',   'content': '영업이익률을 계산해주세요.'},
]

text_think    = tokenizer.apply_chat_template(gen_messages, tokenize=False,
                    add_generation_prompt=True, enable_thinking=True)
text_no_think = tokenizer.apply_chat_template(gen_messages, tokenize=False,
                    add_generation_prompt=True, enable_thinking=False)

print('[Thinking 모드 -- 끝부분]')
print(text_think[-200:])
print()
print('[Non-thinking 모드 -- 끝부분]')
print(text_no_think[-200:])

# 두 모드의 차이
ids_th = set(tokenizer.encode(text_think))
ids_no = set(tokenizer.encode(text_no_think))
diff   = [tokenizer.decode([t]) for t in ids_th - ids_no]
print(f'\nThinking 모드에만 추가된 토큰: {diff}')


[Thinking 모드 -- 끝부분]
<|im_start|>system
당신은 재무 전문가입니다.<|im_end|>
<|im_start|>user
영업이익률을 계산해주세요.<|im_end|>
<|im_start|>assistant


[Non-thinking 모드 -- 끝부분]
<|im_start|>system
당신은 재무 전문가입니다.<|im_end|>
<|im_start|>user
영업이익률을 계산해주세요.<|im_end|>
<|im_start|>assistant
<think>

</think>



Thinking 모드에만 추가된 토큰: []


---

## Step 4. 학습 데이터셋 구축

### 4.1 SFT 데이터셋 설계 원칙

SFT(Supervised Fine-Tuning)는 `(질문, 정답)` 쌍으로 모델이 특정 방식으로 답하도록 학습시킨다.
데이터의 **품질**이 파라미터 수보다 중요하다.

**학습 데이터 포맷 (OpenAI messages 형식):**
```json
{
  "messages": [
    {"role": "system",    "content": "당신은 삼성전자 2024 감사보고서 전문가입니다."},
    {"role": "user",      "content": "2024년 매출액은?"},
    {"role": "assistant", "content": "209조 524억원(209,052,241 백만원)입니다. ..."}
  ]
}
```

### 4.2 생성할 Q&A 유형

| 유형 | 출처 | 예시 | 목표 개수 |
|:---|:---|:---|:---:|
| 직접 수치 조회 | BS · IS · CF | '자산총계는?' | ~25 |
| 비율 · 지표 계산 | IS + BS | '영업이익률은?' | ~15 |
| 전기 대비 증감 | BS · IS | '매출 YoY 성장률은?' | ~15 |
| 텍스트 기반 | sections.json | '감사 의견은?' | ~10 |


In [9]:
# [실습 4-1] 3회차 출력 파일 로드

import pandas as pd, re
from typing import List, Dict, Any, Optional

def load_fin_csv(name):
    path = TABLES_DIR / name
    if not path.exists():
        print(f'  [없음] {name}'); return None
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    return df

bs  = load_fin_csv('2024_BS_재무상태표.csv')
is_ = load_fin_csv('2024_IS_손익계산서.csv')
cf  = load_fin_csv('2024_CF_현금흐름표.csv')
cse = load_fin_csv('2024_CSE_자본변동표.csv')

sections_data = None
if SECTIONS_PATH.exists():
    with open(SECTIONS_PATH, encoding='utf-8') as f:
        sections_data = json.load(f)

for name, df in [('BS', bs), ('IS', is_), ('CF', cf), ('CSE', cse)]:
    print(f'  {name}: {df.shape if df is not None else None}')
print(f'  sections.json: {"로드됨" if sections_data else "없음"}')


  BS: (47, 6)
  IS: (15, 6)
  CF: (31, 6)
  CSE: (17, 7)
  sections.json: 로드됨


In [10]:
# [실습 4-2] 수치 포맷팅 및 Q&A 생성 유틸리티

SYSTEM_PROMPT = (
    '당신은 삼성전자 2024년 별도 재무제표 및 감사보고서를 분석하는 재무 전문가 AI입니다. '
    '수치는 별도 언급이 없는 한 백만원 단위입니다. 정확하고 구체적으로 답변하세요.'
)

def fmt_krw(v):
    if v is None: return '정보 없음'
    s = '-' if v < 0 else ''
    a = abs(v)
    if a >= 1_000_000: return f'{s}{a/1e6:.1f}조원({s}{a:,.0f} 백만원)'
    if a >= 100_000:   return f'{s}{a/1e4:.0f}억원({s}{a:,.0f} 백만원)'
    return f'{s}{a:,.0f} 백만원'

def pct(n, d):
    return '계산불가' if d == 0 else f'{n/d*100:.2f}%'

def yoy(curr, prior):
    if prior == 0: return '전기 0 (비교불가)'
    r = (curr - prior) / abs(prior) * 100
    return f'{abs(r):.1f}% {"증가" if r >= 0 else "감소"}'

def make_qa(q, a):
    return {'messages': [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': q},
        {'role': 'assistant', 'content': a},
    ]}

def get_val(df, kw, col=2):
    if df is None: return None
    mask = df.iloc[:,0].astype(str).str.replace(r'[\s\xa0]+','',regex=True)\
             .str.contains(kw.replace(' ',''), regex=False)
    rows = df[mask]
    if rows.empty: return None
    for c in df.columns[col:]:
        try:
            v = float(rows.iloc[0][c])
            if not pd.isna(v): return v
        except: pass
    return None

print('[포맷팅 테스트]')
for v in [209052241, 12361034, -11526297, 1653766]:
    print(f'  {v:>15,} -> {fmt_krw(v)}')
print(f'영업이익률: {pct(12361034, 209052241)}')
print(f'YoY 매출:   {yoy(209052241, 170374090)}')


[포맷팅 테스트]
      209,052,241 -> 209.1조원(209,052,241 백만원)
       12,361,034 -> 12.4조원(12,361,034 백만원)
      -11,526,297 -> -11.5조원(-11,526,297 백만원)
        1,653,766 -> 1.7조원(1,653,766 백만원)
영업이익률: 5.91%
YoY 매출:   22.7% 증가


In [11]:
# [실습 4-3] 재무상태표(BS) 기반 Q&A 생성

qa_list: List[Dict] = []

if bs is not None:
    a_c  = get_val(bs, '자산총계');      a_p  = get_val(bs, '자산총계', 4)
    ca   = get_val(bs, '유동자산');      nca  = get_val(bs, '비유동자산')
    l_c  = get_val(bs, '부채총계');      l_p  = get_val(bs, '부채총계', 4)
    cl   = get_val(bs, '유동부채')
    eq_c = get_val(bs, '자본총계');      eq_p = get_val(bs, '자본총계', 4)
    re_  = get_val(bs, '이익잉여금');    cash = get_val(bs, '현금및현금성자산')
    inv  = get_val(bs, '재고자산')

    qa_list += [
        make_qa('삼성전자의 2024년 자산총계(총자산)는 얼마인가요?',
            f'자산총계는 {fmt_krw(a_c)}입니다. 전기 {fmt_krw(a_p)} 대비 {yoy(a_c, a_p)}하였습니다.'),
        make_qa('2024년 유동자산과 비유동자산은?',
            f'유동자산 {fmt_krw(ca)}, 비유동자산 {fmt_krw(nca)}입니다. '
            f'비유동자산 비중 약 {pct(nca, a_c)}입니다.'),
        make_qa('2024년 부채비율은?',
            f'부채비율(부채÷자본×100)은 {pct(l_c, eq_c)}입니다. '
            f'부채총계 {fmt_krw(l_c)}, 자본총계 {fmt_krw(eq_c)}입니다.'),
        make_qa('2024년 유동비율은? 재무적으로 안전한가요?',
            f'유동비율(유동자산÷유동부채×100)은 {pct(ca, cl)}입니다. '
            f'100% 이상이면 단기 채무 상환 능력이 있다고 봅니다.'),
        make_qa('2024년 이익잉여금은?',
            f'이익잉여금은 {fmt_krw(re_)}입니다. 설립 이래 누적 순이익에서 배당 등을 차감한 금액입니다.'),
        make_qa('2024년 말 현금및현금성자산 잔액은?',
            f'현금및현금성자산은 {fmt_krw(cash)}입니다. '
            f'전기 대비 크게 감소하였으며 대규모 투자 및 자사주 매입에 기인합니다.'),
        make_qa('2024년 자본총계의 전기 대비 변화는?',
            f'자본총계가 {fmt_krw(eq_p)}에서 {fmt_krw(eq_c)}으로 {yoy(eq_c, eq_p)}하였습니다.'),
        make_qa('2024년 재고자산은?', f'재고자산은 {fmt_krw(inv)}입니다.'),
    ]

print(f'BS Q&A: {len(qa_list)}개')
print(f'  샘플 Q: {qa_list[0]["messages"][1]["content"]}')
print(f'  샘플 A: {qa_list[0]["messages"][2]["content"]}')


BS Q&A: 8개
  샘플 Q: 삼성전자의 2024년 자산총계(총자산)는 얼마인가요?
  샘플 A: 자산총계는 325.0조원(324,966,127 백만원)입니다. 전기 296.9조원(296,857,289 백만원) 대비 9.5% 증가하였습니다.


In [12]:
# [실습 4-4] 손익계산서(IS) 기반 Q&A 생성 (비율·증감 계산 포함)

n_before = len(qa_list)
if is_ is not None:
    r_c  = get_val(is_, '매출액');       r_p  = get_val(is_, '매출액', 4)
    cogs = get_val(is_, '매출원가');     gp   = get_val(is_, '매출총이익')
    op_c = get_val(is_, '영업이익');     op_p = get_val(is_, '영업이익', 4)
    ni_c = get_val(is_, '당기순이익');   ni_p = get_val(is_, '당기순이익', 4)
    tax  = get_val(is_, '법인세비용');   eps  = get_val(is_, '기본주당이익')

    qa_list += [
        make_qa('삼성전자의 2024년 매출액은 얼마인가요?',
            f'2024년 매출액은 {fmt_krw(r_c)}입니다. 전기 {fmt_krw(r_p)} 대비 {yoy(r_c, r_p)}하였습니다.'),
        make_qa('2024년 영업이익과 영업이익률은?',
            f'영업이익은 {fmt_krw(op_c)}, 영업이익률은 {pct(op_c, r_c)}입니다. '
            f'전기 영업손실 {fmt_krw(abs(op_p))} 대비 흑자전환하였습니다.'),
        make_qa('2024년 당기순이익과 순이익률은?',
            f'당기순이익은 {fmt_krw(ni_c)}, 순이익률은 {pct(ni_c, r_c)}입니다. '
            f'전기 {fmt_krw(ni_p)} 대비 {yoy(ni_c, ni_p)}하였습니다.'),
        make_qa('2024년 매출총이익과 매출원가율은?',
            f'매출총이익 {fmt_krw(gp)}, 매출원가율 {pct(cogs, r_c)}입니다.'),
        make_qa('2024년 기본주당이익(EPS)은?',
            f'기본주당이익은 {eps:,.0f}원입니다.' if eps else '주당이익 정보를 찾을 수 없습니다.'),
        make_qa('전기 대비 2024년 실적 개선 여부를 요약해주세요.',
            f'매출액 {yoy(r_c, r_p)}, 영업이익 전기 적자({fmt_krw(abs(op_p))})에서 '
            f'흑자({fmt_krw(op_c)})로 전환, 당기순이익 {fmt_krw(ni_c)} 기록. '
            f'전반적으로 실적이 크게 개선되었습니다.'),
        make_qa('2024년 법인세비용(수익)은?',
            f'법인세비용(수익)은 {fmt_krw(tax)}입니다. '
            f'음수는 이연법인세 수익 발생을 의미합니다.'),
    ]

print(f'IS Q&A: {len(qa_list)-n_before}개 추가 (누적 {len(qa_list)}개)')


IS Q&A: 7개 추가 (누적 15개)


In [13]:
# [실습 4-5] 현금흐름표(CF) 기반 Q&A 생성

n_before = len(qa_list)
if cf is not None:
    op_cf  = get_val(cf, '영업활동');   inv_cf = get_val(cf, '투자활동')
    fin_cf = get_val(cf, '재무활동');   end_c  = get_val(cf, '기말')
    beg_c  = get_val(cf, '기초');      capex  = get_val(cf, '유형자산의취득') or get_val(cf, '유형자산')
    div    = get_val(cf, '배당금의지급') or get_val(cf, '배당금')

    qa_list += [
        make_qa('2024년 영업활동 현금흐름은?',
            f'영업활동 현금흐름은 {fmt_krw(op_cf)}으로 전기 대비 크게 증가하였습니다.'),
        make_qa('2024년 투자활동 현금흐름과 주요 지출은?',
            f'투자활동 현금흐름은 {fmt_krw(inv_cf)}이며 주요 지출은 유형자산 취득(CAPEX) {fmt_krw(capex)}입니다.'),
        make_qa('2024년 재무활동 현금흐름과 주요 내용은?',
            f'재무활동 현금흐름은 {fmt_krw(fin_cf)}으로 배당금 지급 {fmt_krw(div)} 등이 주요 내용입니다.'),
        make_qa('2024년 말 현금 잔액 변화는?',
            f'기초 {fmt_krw(beg_c)}에서 기말 {fmt_krw(end_c)}으로 감소하였습니다.'),
        make_qa('FCF(잉여현금흐름)를 계산해주세요.',
            f'FCF = 영업CF({fmt_krw(op_cf)}) - CAPEX({fmt_krw(abs(capex))}) = '
            f'{fmt_krw(op_cf + (capex or 0))}입니다.'),
    ]

print(f'CF Q&A: {len(qa_list)-n_before}개 추가 (누적 {len(qa_list)}개)')


CF Q&A: 5개 추가 (누적 20개)


In [14]:
# [실습 4-6] sections.json 및 일반 지식 Q&A 생성

n_before = len(qa_list)

if sections_data:
    audit_text = next((s['content'] for s in sections_data.get('sections', [])
                       if s['section_id'] == 'toc_2'), '')
    firms = list(dict.fromkeys(re.findall(r'[가-힣]{2,6}회계법인', audit_text)))
    firm  = firms[0] if firms else '삼정회계법인'
    qa_list += [
        make_qa('삼성전자 2024년 감사를 수행한 회계법인은?',
            f'{firm}이 2024년 재무제표 감사를 수행하였습니다.'),
        make_qa('2024년 감사 의견은 무엇인가요?',
            f'{firm}은 삼성전자 2024년 별도 재무제표에 대해 적정의견을 표명하였습니다. '
            f'K-IFRS에 따라 중요성 관점에서 공정하게 표시되어 있음을 의미합니다.'),
    ]

qa_list += [
    make_qa('이 재무제표는 연결인가요, 별도인가요?',
        '별도 재무제표입니다. 삼성전자 본사 단독 기준이며 종속기업을 포함한 연결 재무제표와 다릅니다.'),
    make_qa('재무제표 작성 기준은 무엇인가요?',
        '한국채택국제회계기준(K-IFRS)에 따라 작성되었습니다.'),
    make_qa('재무제표의 회계기간은?',
        '2024년 1월 1일부터 2024년 12월 31일까지(제56기)입니다.'),
    make_qa('재무제표 수치 단위는?',
        '백만원(KRW million) 단위입니다. 예) 209,052,241 = 약 209조원'),
]

print(f'텍스트/일반 Q&A: {len(qa_list)-n_before}개 추가 (누적 {len(qa_list)}개)')


텍스트/일반 Q&A: 6개 추가 (누적 26개)


In [15]:
# [실습 4-7] 데이터셋 통합, JSONL 저장, HuggingFace Dataset 변환

from datasets import Dataset
import statistics

print(f'총 Q&A 쌍: {len(qa_list)}개')

# JSONL 저장
jsonl_path = OUTPUT_DIR / 'samsung_2024_finetune.jsonl'
with open(jsonl_path, 'w', encoding='utf-8') as f:
    for item in qa_list:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
print(f'JSONL 저장: {jsonl_path} ({jsonl_path.stat().st_size:,} bytes)')

# HuggingFace Dataset
dataset = Dataset.from_list(qa_list)
print(f'Dataset: {dataset}')

# 토큰 길이 분포
lengths = []
for item in qa_list:
    text = tokenizer.apply_chat_template(item['messages'],
        tokenize=False, add_generation_prompt=False, enable_thinking=False)
    lengths.append(len(tokenizer.encode(text)))

print(f'\n[토큰 길이 통계]')
print(f'  최솟값: {min(lengths)}')
print(f'  최댓값: {max(lengths)}')
print(f'  평균:   {statistics.mean(lengths):.1f}')
print(f'  중앙값: {statistics.median(lengths)}')

print('\n[샘플 3개]')
for i, item in enumerate(qa_list[:3]):
    q = item['messages'][1]['content']
    a = item['messages'][2]['content']
    print(f'  [{i+1}] Q: {q}')
    print(f'       A: {(a[:80]+"...") if len(a)>80 else a}')
    print()


총 Q&A 쌍: 26개
JSONL 저장: ../output/finetune/samsung_2024_finetune.jsonl (13,530 bytes)
Dataset: Dataset({
    features: ['messages'],
    num_rows: 26
})

[토큰 길이 통계]
  최솟값: 115
  최댓값: 223
  평균:   156.8
  중앙값: 157.5

[샘플 3개]
  [1] Q: 삼성전자의 2024년 자산총계(총자산)는 얼마인가요?
       A: 자산총계는 325.0조원(324,966,127 백만원)입니다. 전기 296.9조원(296,857,289 백만원) 대비 9.5% 증가하였습니다.

  [2] Q: 2024년 유동자산과 비유동자산은?
       A: 유동자산 82.3조원(82,320,322 백만원), 비유동자산 242.6조원(242,645,805 백만원)입니다. 비유동자산 비중 약 74.67...

  [3] Q: 2024년 부채비율은?
       A: 부채비율(부채÷자본×100)은 37.47%입니다. 부채총계 88.6조원(88,569,470 백만원), 자본총계 236.4조원(236,396,65...



---

## Step 5. QLoRA 파인튜닝

### 5.1 양자화(Quantization)란

모델 가중치는 기본적으로 float32(4 bytes) 또는 bfloat16(2 bytes)으로 저장된다.
양자화는 이를 더 낮은 비트 수로 압축하여 메모리를 절약한다.

```
float32 (4 bytes/값)  -->  4-bit NF4 (0.5 bytes/값)
메모리: 약 8배 절감

Qwen3-0.6B 기준:
  bfloat16 : ~1.2 GB
  4-bit    : ~0.3 GB  <-- Colab / 로컬 CUDA
  float16  : ~1.2 GB  <-- Mac MPS (양자화 없음)
```

### 5.2 환경별 학습 전략

| 환경 | 모델 로드 | 옵티마이저 | 배치 | max_seq |
|:---|:---|:---|:---:|:---:|
| **CUDA (Colab T4 / 로컬)** | 4-bit NF4 | paged_adamw_8bit | 2 | 1024 |
| **Mac (MPS)** | float16 | adamw_torch | 1 | 512 |
| **CPU** | float32 | adamw_torch | 1 | 256 |

### 5.3 LoRA (Low-Rank Adaptation)

전체 가중치를 업데이트하는 대신, 각 가중치 행렬에 **작은 두 행렬(A, B)의 곱**을 추가한다.

```
원래 가중치: W             (freeze -- 업데이트 안 함)
추가 어댑터: alpha x (B x A)  (A, B 만 학습 -- 전체 파라미터의 0.1~1%)
실제 출력:   W·x + alpha·B·A·x

rank r=16 이면: A는 (d x 16), B는 (16 x d)
```

### 5.4 QLoRA = 4-bit 양자화 + LoRA (CUDA 전용)

```
1. 베이스 모델을 4-bit NF4 로 로드       (메모리 절약)
2. LoRA 어댑터를 bfloat16 으로 초기화    (정밀도 유지)
3. 4-bit 가중치는 freeze, LoRA 만 역전파
4. 추론 시: 4-bit 가중치 + LoRA 합산 적용
```

Mac / CPU 환경에서는 4-bit 없이 **LoRA 만** 적용한다 (Full LoRA).

### 5.5 Qwen3 LoRA 적용 모듈

| 모듈 | 역할 |
|:---|:---|
| `q_proj` `k_proj` `v_proj` | Self-Attention QKV 프로젝션 |
| `o_proj` | Attention 출력 프로젝션 |
| `gate_proj` `up_proj` `down_proj` | MLP (Feed-Forward Network) |


In [16]:
# [실습 5-1] 환경에 맞는 모델 로드
# CUDA  : 4-bit NF4 양자화 (QLoRA)
# Mac   : float16 전체 로드 (MPS)
# CPU   : float32 전체 로드

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

if ENV_CONFIG['use_4bit']:
    # ── CUDA 환경: 4-bit QLoRA ──────────────────────────────
    print('[CUDA] 4-bit NF4 양자화 모델 로드...')
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=ENV_CONFIG['compute_dtype'],
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map='auto',
        torch_dtype=ENV_CONFIG['torch_dtype'],
    )
elif ENV_CONFIG['device'] == 'mps':
    # ── Mac MPS 환경: float16 전체 로드 ─────────────────────
    print('[Mac MPS] float16 모델 로드 (4-bit 양자화 미지원)...')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map='mps',
    )
else:
    # ── CPU 환경: float32 전체 로드 ─────────────────────────
    print('[CPU] float32 모델 로드 (학습 속도 매우 느림, 테스트 전용)...')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
        device_map='cpu',
    )

# ── 메모리 정보 출력 ────────────────────────────────────────
if ENV_CONFIG['device'] == 'cuda':
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU 메모리: {used:.2f} GB / {total:.1f} GB')
elif ENV_CONFIG['device'] == 'mps':
    print('MPS 디바이스로 로드 완료 (통합 메모리 사용)')

total_p = sum(p.numel() for p in model.parameters())
print(f'파라미터: {total_p/1e6:.1f}M')
print(f'실제 로드 dtype: {next(model.parameters()).dtype}')


[Mac MPS] float16 모델 로드 (4-bit 양자화 미지원)...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

MPS 디바이스로 로드 완료 (통합 메모리 사용)
파라미터: 596.0M
실제 로드 dtype: torch.float16


In [17]:
# [실습 5-2] LoRA 어댑터 설정 및 적용

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if ENV_CONFIG['use_4bit']:
    # 4-bit 양자화 모델은 kbit 학습 전처리 필요 (CUDA 전용)
    model = prepare_model_for_kbit_training(model)
    print('[CUDA] kbit 학습 전처리 완료')
else:
    print(f'[{ENV_CONFIG["device"].upper()}] kbit 전처리 생략 (4-bit 미사용)')

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f'\n학습 파라미터: {trainable:,} / 전체 {total_p:,} ({trainable/total_p*100:.3f}%)')


[MPS] kbit 전처리 생략 (4-bit 미사용)
'NoneType' object has no attribute 'cadam32bit_grad_fp32'


/opt/homebrew/Caskroom/miniforge/base/envs/llm_studio/lib/python3.11/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650

학습 파라미터: 10,092,544 / 전체 606,142,464 (1.665%)


In [21]:
# [실습 5-3] SFTTrainer 구성 — 환경별 설정 분기
# - CUDA : paged_adamw_8bit, bf16/fp16, gradient_checkpointing 가능
# - Mac  : adamw_torch, fp16 (MPS bf16 불안정), gradient_checkpointing 제한
# - CPU  : adamw_torch, fp16 불가

from trl import SFTTrainer, SFTConfig

if ENV_CONFIG['device'] == 'cuda':
    sft_config = SFTConfig(
        output_dir=str(OUTPUT_DIR / 'checkpoints'),
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,       # 실효 배치 = 8
        gradient_checkpointing=True,         # VRAM 절약
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        weight_decay=0.01,
        optim='paged_adamw_8bit',            # bitsandbytes 8-bit 옵티마이저
        logging_steps=10,
        save_steps=50,
        save_total_limit=2,
        report_to='none',
        fp16=not ENV_CONFIG['bf16_ok'],
        bf16=ENV_CONFIG['bf16_ok'],
    )
    print('[CUDA] SFTConfig — paged_adamw_8bit, gradient_checkpointing=True')

elif ENV_CONFIG['device'] == 'mps':
    sft_config = SFTConfig(
        output_dir=str(OUTPUT_DIR / 'checkpoints'),
        num_train_epochs=3,
        per_device_train_batch_size=1,       # MPS 는 배치 1 이 안정적
        gradient_accumulation_steps=8,       # 실효 배치 = 8
        gradient_checkpointing=False,        # MPS 에서 불안정할 수 있음
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        weight_decay=0.01,
        optim='adamw_torch',                 # MPS 호환 표준 옵티마이저
        logging_steps=10,
        save_steps=50,
        save_total_limit=2,
        report_to='none',
        fp16=True,                           # MPS: float16 학습
        bf16=False,
    )
    print('[Mac MPS] SFTConfig — adamw_torch, fp16=True, batch=1')

else:  # CPU
    sft_config = SFTConfig(
        output_dir=str(OUTPUT_DIR / 'checkpoints'),
        num_train_epochs=1,                  # CPU: 에폭 1 로 제한
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        gradient_checkpointing=False,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        weight_decay=0.01,
        optim='adamw_torch',                 
        logging_steps=5,
        save_steps=20,
        save_total_limit=1,
        report_to='none',
        fp16=False,
        bf16=False,
    )
    print('[CPU] SFTConfig — adamw_torch, epoch=1 (테스트 전용)')

tokenizer.padding_side = 'right'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

steps = len(dataset) * sft_config.num_train_epochs \
        // (sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps)
print(f'SFTTrainer 초기화 완료')
print(f'학습 데이터: {len(dataset)}개 | 예상 스텝: {steps:.0f}')
print(f'환경 요약: device={ENV_CONFIG["device"]}, 4-bit={ENV_CONFIG["use_4bit"]}, optim={ENV_CONFIG["optim"]}')


[Mac MPS] SFTConfig — adamw_torch, fp16=True, batch=1


Tokenizing train dataset:   0%|          | 0/26 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/26 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


SFTTrainer 초기화 완료
학습 데이터: 26개 | 예상 스텝: 9
환경 요약: device=mps, 4-bit=False, optim=adamw_torch


In [22]:
# [실습 5-4] 파인튜닝 학습 실행
# 예상 시간: Colab T4 기준 약 5~10분 (데이터 ~65개, 3 에폭)

print('학습 시작...')
result = trainer.train()

print('\n[학습 결과]')
print(f'  총 시간:   {result.metrics.get("train_runtime", 0):.1f}초')
print(f'  최종 Loss: {result.metrics.get("train_loss", 0):.4f}')

# LoRA 어댑터 저장
adapter_path = OUTPUT_DIR / 'lora_adapter'
model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))
print(f'\nLoRA 어댑터 저장: {adapter_path}')


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


학습 시작...


/opt/homebrew/Caskroom/miniforge/base/envs/llm_studio/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,1.931500



[학습 결과]
  총 시간:   25.4초
  최종 Loss: 1.7482

LoRA 어댑터 저장: ../output/finetune/lora_adapter


---

## Step 6. 추론 및 평가

파인튜닝된 모델로 추론을 수행하고 학습 전후를 비교한다.
Qwen3 의 Thinking / Non-thinking 모드를 실제로 실습한다.

### 권장 샘플링 파라미터

| 모드 | Temperature | TopP | TopK |
|:---|:---:|:---:|:---:|
| Thinking | 0.6 | 0.95 | 20 |
| Non-thinking | 0.7 | 0.8 | 20 |


In [23]:
# [실습 6-1] 추론 유틸리티 함수

def generate_answer(question, model_obj, tok, thinking=False, max_new_tokens=512):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': question},
    ]
    text = tok.apply_chat_template(messages, tokenize=False,
               add_generation_prompt=True, enable_thinking=thinking)
    inputs = tok([text], return_tensors='pt').to(model_obj.device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        temperature=0.6 if thinking else 0.7,
        top_p=0.95 if thinking else 0.8,
        top_k=20, do_sample=True,
        pad_token_id=tok.eos_token_id,
    )
    with torch.no_grad():
        generated = model_obj.generate(**inputs, **gen_kwargs)

    output_ids = generated[0][len(inputs.input_ids[0]):].tolist()

    if thinking:
        # </think> 토큰(151668) 으로 thinking / answer 분리
        try:
            idx = len(output_ids) - output_ids[::-1].index(151668)
        except ValueError:
            idx = 0
        return {
            'thinking': tok.decode(output_ids[:idx], skip_special_tokens=True).strip(),
            'answer':   tok.decode(output_ids[idx:], skip_special_tokens=True).strip(),
        }
    return {'answer': tok.decode(output_ids, skip_special_tokens=True).strip()}

print('추론 함수 준비 완료')


추론 함수 준비 완료


In [24]:
# [실습 6-2] 파인튜닝 후 모델 추론 -- Non-thinking 모드

test_qs = [
    '삼성전자의 2024년 매출액은 얼마인가요?',
    '2024년 영업이익률은 얼마인가요?',
    '2024년 부채비율은 얼마인가요?',
    '2024년 영업활동 현금흐름은?',
]

print('=' * 65)
print('[파인튜닝 후 -- Non-thinking 모드]')
print('=' * 65)
for q in test_qs:
    r = generate_answer(q, model, tokenizer, thinking=False, max_new_tokens=256)
    print(f'Q: {q}')
    print(f'A: {r["answer"]}')
    print('-' * 65)


[파인튜닝 후 -- Non-thinking 모드]
Q: 삼성전자의 2024년 매출액은 얼마인가요?
A: 2024년 매출액은 1.22조원(12,209,760,000 백만원)입니다. 별도 언급이 없는 한 백만원 단위입니다.
-----------------------------------------------------------------
Q: 2024년 영업이익률은 얼마인가요?
A: 2024년 영업이익률은 26.5%입니다. (26.5 백만원 단위)
-----------------------------------------------------------------
Q: 2024년 부채비율은 얼마인가요?
A: 2024년 삼성전자 부채비율은 32.18%입니다. (재무제표 2024년 1월 1일부터 2024년 12월 31일까지)
-----------------------------------------------------------------
Q: 2024년 영업활동 현금흐름은?
A: 2024년 영업활동 현금흐름은 1.8조원(1.83万亿원)입니다.
-----------------------------------------------------------------


In [25]:
# [실습 6-3] Thinking 모드 실습 -- 복합 재무 분석 질문

complex_q = ('삼성전자 2024년 재무 건전성을 '
             '부채비율, 유동비율, 영업이익률 기준으로 종합 평가해주세요.')

print('=' * 65)
print(f'Q: {complex_q}\n')

r = generate_answer(complex_q, model, tokenizer, thinking=True, max_new_tokens=1024)

print('[사고 과정 <think>]')
t = r.get('thinking', '')
print((t[:500] + '...') if len(t) > 500 else (t or '(없음)'))
print('\n[최종 답변]')
print(r['answer'])


Q: 삼성전자 2024년 재무 건전성을 부채비율, 유동비율, 영업이익률 기준으로 종합 평가해주세요.

[사고 과정 <think>]
<think>
Okay, let's start by recalling the key metrics for financial health. The user mentioned the debt-to-EBITDA ratio and liquidity ratio, as well as the operating profit margin. I need to calculate each of these and compare them to industry benchmarks to give a comprehensive evaluation.

First, the debt-to-EBITDA ratio. The company's total debt is calculated as total liabilities divided by EBITDA. For 2024, I should look up the exact numbers. Let's say the total liabilities are 5.2 trillion ...

[최종 답변]
- **부채비율 (자금비율): 2.9** (industry avg: 3.5) → 부채 관리 잘.  
- **유동비율: 1.8** (industry avg: 0.7-1.2) → 금융적 안정성 유지.  
- **영업이익률: 33.3%** (industry avg: 30-35) → 영업 효율성 우수.  

**결과:** 별도 언급 없이 재무 건전성은 높은 것으로 평가됩니다.


In [28]:
# [실습 6-4] 파인튜닝 전후 비교
# LoRA 어댑터를 disable 하면 베이스 모델 응답을 확인할 수 있다

compare_q = '삼성전자의 2024년 매출액은 얼마인가요?'
print(f'Q: {compare_q}')
print('=' * 65)

model.eval()
r_ft = generate_answer(compare_q, model, tokenizer, thinking=False, max_new_tokens=256)

with model.disable_adapter():
    r_base = generate_answer(compare_q, model, tokenizer, thinking=False, max_new_tokens=256)

print('\n[베이스 모델 (파인튜닝 전)]')
print(r_base['answer'])
print('\n[파인튜닝 모델 (LoRA 적용)]')
print(r_ft['answer'])
print('\n[정답]  209조 524억원(209,052,241 백만원) -- 전기 대비 22.7% 증가')


Q: 삼성전자의 2024년 매출액은 얼마인가요?

[베이스 모델 (파인튜닝 전)]
2024년 삼성전자 매출액은 **1.6 trillion won**입니다.

[파인튜닝 모델 (LoRA 적용)]
2024년 삼성전자 매출액은 2.5조원원입니다. 별도 언급이 없는 한 백만원 단위입니다.

[정답]  209조 524억원(209,052,241 백만원) -- 전기 대비 22.7% 증가


In [29]:
# [실습 6-5] /no_think 소프트 스위치 실습
# enable_thinking=True 상태에서도 프롬프트 끝에 /no_think 를 붙이면 사고 없이 답변한다

import time

base_q = '2024년 영업이익률을 계산해주세요.'

for suffix, label in [('', 'thinking 기본'), (' /no_think', '/no_think 스위치')]:
    q = base_q + suffix
    t0 = time.time()
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': q}]
    text = tokenizer.apply_chat_template(messages, tokenize=False,
               add_generation_prompt=True, enable_thinking=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=300,
                             temperature=0.6, top_p=0.95, top_k=20, do_sample=True,
                             pad_token_id=tokenizer.eos_token_id)
    ids = out[0][len(inputs.input_ids[0]):].tolist()
    try:
        idx = len(ids) - ids[::-1].index(151668)
    except ValueError:
        idx = 0
    answer = tokenizer.decode(ids[idx:], skip_special_tokens=True).strip()
    elapsed = time.time() - t0

    print(f'[{label}] ({elapsed:.1f}초, thinking 토큰={idx})')
    print(f'  A: {answer[:200]}')
    print()


[thinking 기본] (1.8초, thinking 토큰=3)
  A: 2024년 영업이익률은 3,839,400 백원(3.84%)입니다.

[/no_think 스위치] (1.1초, thinking 토큰=3)
  A: 2024년 영업이익률은 34.3%입니다.



---

## 핵심 개념 정리

| 개념 | 설명 |
|:---|:---|
| **토크나이저** | 텍스트 -> 정수 ID 시퀀스 변환. BPE 알고리즘 사용 |
| **Chat Template** | system/user/assistant 역할을 모델 입력 형식으로 변환 |
| **Thinking 모드** | `<think>...</think>` 사고 과정 후 답변. Qwen3 고유 기능 |
| **4-bit 양자화** | 가중치를 4-bit 로 압축하여 메모리 최대 8배 절약 |
| **QLoRA** | 4-bit 베이스 + bfloat16 LoRA 어댑터. 소비자 GPU 에서 대형 모델 파인튜닝 |
| **LoRA rank (r)** | 적응 행렬 차원. r 클수록 표현력 높고 메모리 증가. 보통 8~64 |
| **SFTTrainer** | TRL 의 파인튜닝 고수준 API. Chat Template 자동 처리 |

---

## 다음 단계

- **RAG 파이프라인**: 재무 데이터 -> 벡터 DB -> 검색 기반 답변 생성
- **DPO 정렬**: 선호 답변 쌍으로 사람 선호도 학습
- **평가 자동화**: 재무 도메인 벤치마크 구성 및 자동 채점
